In [1]:
import os
import json
import cv2
import numpy as np

# 한글 경로 우회해서 이미지 읽고 쓰는 실무용 함수
def imread_korean(path):
    try:
        img_array = np.fromfile(path, np.uint8)
        return cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    except: return None

def imwrite_korean(path, img):
    try:
        ext = os.path.splitext(path)[1]
        result, encoded_img = cv2.imencode(ext, img)
        if result:
            with open(path, mode='w+b') as f:
                encoded_img.tofile(f)
    except Exception as e: pass

def process_and_crop(json_dir, img_dir, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
    
    count = 0
    for j_file in json_files:
        with open(os.path.join(json_dir, j_file), 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        img_filename = data['images'][0]['file_name']
        
        # 라벨링 안 된 쓰레기 데이터 방어 로직
        if 'annotations' not in data or len(data['annotations']) == 0:
            continue
            
        bbox = data['annotations'][0]['bbox'] # [x, y, width, height]
        img_path = os.path.join(img_dir, img_filename)
        
        image = imread_korean(img_path)
        if image is None: continue
            
        # 좌표 정수 변환 및 크롭 (실무 1티어 방어 코드)
        x, y, w, h = map(int, bbox)
        y_end = min(y + h, image.shape[0])
        x_end = min(x + w, image.shape[1])
        cropped_img = image[y:y_end, x:x_end]
        
        if cropped_img.size > 0:
            resized_img = cv2.resize(cropped_img, (224, 224)) # 딥러닝 국룰 사이즈
            save_path = os.path.join(save_dir, img_filename)
            imwrite_korean(save_path, resized_img)
            count += 1
            
    print(f"[{save_dir}] 완료: {count}장 변환됨")

# ================= 네 경로 매핑 =================
base_src = r"D:\104.부품 품질 검사 영상 데이터_선박-해양플랜드_고도화_LNG탱크 품질 검사 영상 데이터\3.개방데이터\1.데이터"
base_dst = r"D:\DL\Joint_dataset"

# (json폴더, 이미지폴더, 저장할폴더) 쌍으로 묶음
tasks = [
    # Train 데이터
    (rf"{base_src}\Training\02.라벨링데이터\TL_용접_용접불량_조인트", rf"{base_src}\Training\01.원천데이터\TS_용접_용접불량_조인트", rf"{base_dst}\train\bad_weld"),
    (rf"{base_src}\Training\02.라벨링데이터\TL_용접_용접블루홀_조인트", rf"{base_src}\Training\01.원천데이터\TS_용접_용접블루홀_조인트", rf"{base_dst}\train\blowhole"),
    (rf"{base_src}\Training\02.라벨링데이터\TL_용접_용접양품_조인트", rf"{base_src}\Training\01.원천데이터\TS_용접_용접양품_조인트", rf"{base_dst}\train\good"),
    
    # Validation 데이터
    (rf"{base_src}\Validation\02.라벨링데이터\VL_용접_용접불량_조인트", rf"{base_src}\Validation\01.원천데이터\VS_용접_용접불량_조인트", rf"{base_dst}\val\bad_weld"),
    (rf"{base_src}\Validation\02.라벨링데이터\VL_용접_용접블루홀_조인트", rf"{base_src}\Validation\01.원천데이터\VS_용접_용접블루홀_조인트", rf"{base_dst}\val\blowhole"),
    (rf"{base_src}\Validation\02.라벨링데이터\VL_용접_용접양품_조인트", rf"{base_src}\Validation\01.원천데이터\VS_용접_용접양품_조인트", rf"{base_dst}\val\good")
]

for json_dir, img_dir, save_dir in tasks:
    process_and_crop(json_dir, img_dir, save_dir)
    
print("전체 데이터 전처리 싹 다 끝났다 이기야 ㅋㅋㅋ")

[D:\DL\Joint_dataset\train\bad_weld] 완료: 5383장 변환됨
[D:\DL\Joint_dataset\train\blowhole] 완료: 8946장 변환됨
[D:\DL\Joint_dataset\train\good] 완료: 5497장 변환됨
[D:\DL\Joint_dataset\val\bad_weld] 완료: 668장 변환됨
[D:\DL\Joint_dataset\val\blowhole] 완료: 1066장 변환됨
[D:\DL\Joint_dataset\val\good] 완료: 699장 변환됨
전체 데이터 전처리 싹 다 끝났다 이기야 ㅋㅋㅋ
